# imports

In [33]:
import torch

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



CUDA disponible: True
GPU: NVIDIA GeForce RTX 4060
Device: cuda


In [34]:
from pathlib import Path
import ast
import json

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

# detectar raíz del proyecto

In [35]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("No pude encontrar la raíz del proyecto.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
WORKING_DIR = DATA_DIR / "working"
MANIFESTS_DIR = DATA_DIR / "manifests"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("WORKING_DIR:", WORKING_DIR)
print("MANIFESTS_DIR:", MANIFESTS_DIR)

PROJECT_ROOT: /mnt/d/Universidad/analitica/proyecto_analitica2
DATA_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data
WORKING_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working
MANIFESTS_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests


# cargar manifiestos necesarios

In [36]:
manifest_full_path = MANIFESTS_DIR / "manifest_nih_final_with_paths.csv"
subset_path = WORKING_DIR / "nih" / "subsets" / "nih_subset_baseline_with_paths.csv"

manifest_full = pd.read_csv(manifest_full_path)
subset_df = pd.read_csv(subset_path)

print("subset_df:", subset_df.shape)
print("\nConteo por split:")
print(subset_df["split_final"].value_counts())

subset_df: (7000, 19)

Conteo por split:
split_final
train    5000
val      1000
test     1000
Name: count, dtype: int64


# parsear labels_list

In [37]:
def safe_parse_labels(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    x = str(x).strip()
    if not x:
        return []
    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass
    return [label.strip() for label in x.split("|") if label.strip()]

manifest_full["labels_list"] = manifest_full["labels_list"].apply(safe_parse_labels)
subset_df["labels_list"] = subset_df["labels_list"].apply(safe_parse_labels)

subset_df[["image_name", "labels_list"]].head()

,image_name,labels_list
0,00022245_021.png,[No Finding]
1,00019544_000.png,[No Finding]
2,00009673_001.png,[Pleural_Thickening]
3,00018103_001.png,[No Finding]
4,00017799_000.png,[Nodule]


# construir vocabulario de etiquetas

In [38]:
all_labels = sorted({
    label
    for labels in manifest_full["labels_list"]
    for label in labels
})

label_to_idx = {label: i for i, label in enumerate(all_labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

print("Etiquetas encontradas:")
print(all_labels)
print("\nNúmero de etiquetas:", len(all_labels))

Etiquetas encontradas:
['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']

Número de etiquetas: 15


# guardar mapeo de etiquetas

In [39]:
label_map_dir = WORKING_DIR / "nih" / "meta"
label_map_dir.mkdir(parents=True, exist_ok=True)

label_map_path = label_map_dir / "nih_label_to_idx.json"
with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump(label_to_idx, f, ensure_ascii=False, indent=2)

print("Mapa de etiquetas guardado en:", label_map_path)

Mapa de etiquetas guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/meta/nih_label_to_idx.json


# función multihot

In [40]:
def labels_to_multihot(labels, label_to_idx):
    vec = torch.zeros(len(label_to_idx), dtype=torch.float32)
    for label in labels:
        if label in label_to_idx:
            vec[label_to_idx[label]] = 1.0
    return vec

# transforms básicos

In [41]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

# clase Dataset

In [42]:
class NIHSubsetDataset(Dataset):
    def __init__(self, dataframe, label_to_idx, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.label_to_idx = label_to_idx
        self.transform = transform

        required_cols = ["file_path", "labels_list", "split_final", "image_name"]
        for col in required_cols:
            if col not in self.df.columns:
                raise ValueError(f"Falta la columna requerida: {col}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["file_path"])
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        labels = row["labels_list"]
        target = labels_to_multihot(labels, self.label_to_idx)

        return {
            "image": image,
            "target": target,
            "image_name": row["image_name"],
            "file_path": str(image_path),
            "split_final": row["split_final"],
        }

# separar train, val y test

In [43]:
train_df = subset_df[subset_df["split_final"] == "train"].copy()
val_df = subset_df[subset_df["split_final"] == "val"].copy()
test_df = subset_df[subset_df["split_final"] == "test"].copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (5000, 19)
val: (1000, 19)
test: (1000, 19)


# revisar rutas faltantes

In [44]:
def count_missing_files(df):
    return (~df["file_path"].apply(lambda x: Path(x).exists())).sum()

print("Faltantes en train:", count_missing_files(train_df))
print("Faltantes en val:", count_missing_files(val_df))
print("Faltantes en test:", count_missing_files(test_df))

Faltantes en train: 0
Faltantes en val: 0
Faltantes en test: 0


# crear datasets

In [45]:
train_dataset = NIHSubsetDataset(train_df, label_to_idx, transform=train_transform)
val_dataset = NIHSubsetDataset(val_df, label_to_idx, transform=eval_transform)
test_dataset = NIHSubsetDataset(test_df, label_to_idx, transform=eval_transform)

print("Tamaños de datasets:")
print("train:", len(train_dataset))
print("val:", len(val_dataset))
print("test:", len(test_dataset))

Tamaños de datasets:
train: 5000
val: 1000
test: 1000


# crear dataloaders

In [46]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Dataloaders creados correctamente.")

Dataloaders creados correctamente.


# inspección de una muestra

In [47]:
sample = train_dataset[0]

print("Claves:", sample.keys())
print("Shape imagen:", sample["image"].shape)
print("Shape target:", sample["target"].shape)
print("image_name:", sample["image_name"])
print("split_final:", sample["split_final"])

Claves: dict_keys(['image', 'target', 'image_name', 'file_path', 'split_final'])
Shape imagen: torch.Size([3, 224, 224])
Shape target: torch.Size([15])
image_name: 00022245_021.png
split_final: train


# inspección de un batch

In [48]:
batch = next(iter(train_loader))

print("Tipo de batch:", type(batch))
print("Shape batch imágenes:", batch["image"].shape)
print("Shape batch targets:", batch["target"].shape)
print("Primeros image_name:", batch["image_name"][:5])

Tipo de batch: <class 'dict'>
Shape batch imágenes: torch.Size([32, 3, 224, 224])
Shape batch targets: torch.Size([32, 15])
Primeros image_name: ['00010585_038.png', '00027022_000.png', '00009649_002.png', '00014290_009.png', '00030129_003.png']
